In [2]:
import numpy as np
import pandas as pd
import gspread
from google.auth import default
from google.colab import auth
from sklearn.preprocessing import MinMaxScaler

# Google Sheets 인증 및 데이터 불러오기
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
spreadsheet_id = "1MI-HWfwR5-Dp-A8dJ4DfOVGGau-9QdysfvsQHvZ5VY0"
spreadsheet = gc.open_by_key(spreadsheet_id)
worksheet = spreadsheet.sheet1
data = worksheet.get_all_values()
df = pd.DataFrame(data[1:], columns=data[0])

# 데이터 타입 변환
df = df.apply(pd.to_numeric, errors='coerce')

# X (입력 변수)와 y (타겟 변수) 분리
X = df.drop(columns=['Purchase']).values
y = df['Purchase'].values.reshape(-1, 1)

# 데이터 정규화 (MinMaxScaler 적용)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# =============== 로지스틱 회귀 구현 =================
learning_rate = 0.01       # 학습률
epochs = 1000   # 학습 횟수
m, n = X_scaled.shape  # 훈련 데이터 수, 가중치 수

# 가중치 초기화
W = np.zeros((n + 1, 1))  # bias 포함

# 가중치 학습 시 bias도 같이 처리하기 위한 더미(dummy) 데이터를 X에 추가
X_bias = np.hstack([np.ones((m, 1)), X_scaled])

# 시그모이드 함수
sigmoid = lambda z: 1 / (1 + np.exp(-z))

# 경사하강법 학습
for i in range(epochs):
    y_pred = sigmoid(np.dot(X_bias, W))  # 선형회귀값을 0~1 사이의 값으로 변환
    cost = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))  # 손실함수 크로스 엔트로피
    gradient = np.dot(X_bias.T, (y_pred - y)) / m  # 기울기 계산
    W -= learning_rate * gradient  # 가중치 업데이트

    # 100 에포크마다 cost 출력
    if (i + 1) % 100 == 0: print(f"Epoch {i + 1}: Cost = {cost:.4f}")

# 전체 데이터에 대한 최종 예측
y_pred_final = (sigmoid(np.dot(X_bias, W)) >= 0.5).astype(int)

# 학습 데이터에서의 정확도 출력
accuracy = np.mean(y_pred_final == y)
print(f'\n 학습된 모델의 예측 정확도: {accuracy:.4f}')


#======================== 아래 코드는 책에서는 생략 ============================
# 새로운 사용자 데이터 입력 및 예측
print(f"\n\n▶ 새로운 사용자의 상품 구매 가능성 예측하기\n")

user_info = [[
    int(input("나이: ")),
    int(input("성별 (남성:0, 여성:1): ")),
    int(input("방문한 페이지 수 (예: 3, 8, 21): ")),
    float(input("사이트에서 머문 시간 (단위:분): ")),
    int(input("이전 구매 횟수 (단위:회): ")),
    int(input("광고 클릭 여부 (클릭 안 함:0, 클릭함:1): "))
]]

# 입력값을 DataFrame으로 변환
new_user_df = pd.DataFrame(user_info, columns=df.drop(columns=['Purchase']).columns)

# 입력값 정규화
new_user_scaled = scaler.transform(new_user_df.values)
new_user_bias = np.hstack([np.ones((1, 1)), new_user_scaled])

# 예측 수행
user_pred = sigmoid(np.dot(new_user_bias, W))

# 예측 결과 출력
purchase_prob = user_pred[0, 0]
print(f"\n시그모이드 함수의 산출값: {purchase_prob:.2f}")
if purchase_prob >= 0.5:
    print("▶ 이 사용자는 상품을 구매할 가능성이 높습니다.")
else:
    print("▶ 이 사용자는 상품을 구매할 가능성이 낮습니다.")

Epoch 100: Cost = 0.4922
Epoch 200: Cost = 0.4202
Epoch 300: Cost = 0.3900
Epoch 400: Cost = 0.3755
Epoch 500: Cost = 0.3677
Epoch 600: Cost = 0.3633
Epoch 700: Cost = 0.3605
Epoch 800: Cost = 0.3587
Epoch 900: Cost = 0.3574
Epoch 1000: Cost = 0.3564

 학습된 모델의 예측 정확도: 0.8735


▶ 새로운 사용자의 상품 구매 가능성 예측하기

나이: 30
성별 (남성:0, 여성:1): 1
방문한 페이지 수 (예: 3, 8, 21): 10
사이트에서 머문 시간 (단위:분): 20
이전 구매 횟수 (단위:회): 5
광고 클릭 여부 (클릭 안 함:0, 클릭함:1): 1

시그모이드 함수의 산출값: 0.90
▶ 이 사용자는 상품을 구매할 가능성이 높습니다.
